# Fase 0 — Estructura de mercado (legs) + Order Blocks

Tercera iteración del concepto de swings (ver `docs/rule_decisions.md` para el historial completo):
1. Ventana fija de N velas — marcaba ruido, no respetaba HH/HL ni LH/LL.
2. Confirmación por reversión de ATR — corregía la secuencia, pero el umbral de tamaño era arbitrario y se comía swings reales.
3. **Esta versión**: pivots locales (cada vela que es el extremo entre sus dos vecinas inmediatas) fusionados hasta alternar perfectamente. Sin ningún parámetro de tamaño — solo la forma real del precio.

Para cada temporalidad se toman los últimos N swings confirmados **desde hoy hacia atrás** (no fechas arbitrarias del histórico), con los OBs correspondientes marcados.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd().name == "00_rule_spec") else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import yaml
import numpy as np
import pandas as pd
import mplfinance as mpf
from smartmoneyconcepts import smc

from trading_ai.data import store
from trading_ai.structure.swings import detect_structural_swings

Thank you for using SmartMoneyConcepts! ⭐ Please show your support by giving a star on the GitHub repository: https://github.com/joshyattridge/smart-money-concepts


In [2]:
# --- Configuración: ajusta esto entre iteraciones ---
SYMBOL = "EURUSD"
with open(PROJECT_ROOT / "config" / "timeframes.yaml", encoding="utf-8") as f:
    TIMEFRAMES = yaml.safe_load(f)["timeframes"]

# Legs consecutivos a mostrar por temporalidad, tomando los MÁS RECIENTES
# (desde hoy hacia atrás, no fechas arbitrarias del histórico).
LEGS_PER_TIMEFRAME = {tf: 6 for tf in TIMEFRAMES}
LEGS_PER_TIMEFRAME["1D"] = 3

PAD_BARS_FRACTION = 0.15  # padding a cada lado, como fracción del ancho de la ventana

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "00_rule_spec" / "output" / "market_structure"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def render_structure_window(
    df: pd.DataFrame,
    swing_points: pd.DataFrame,
    obs: pd.DataFrame,
    lo: int,
    hi: int,
    title: str,
    out_path: Path,
) -> None:
    """Renderiza velas[lo:hi] con los swings (legs) y OBs que caigan dentro."""
    window = df.iloc[lo:hi]

    swing_high = pd.Series(index=window.index, dtype=float)
    swing_low = pd.Series(index=window.index, dtype=float)
    window_swings = swing_points[(swing_points.index >= window.index.min()) & (swing_points.index <= window.index.max())]
    for ts, row in window_swings.iterrows():
        if row["HighLow"] == 1:
            swing_high.loc[ts] = row["Level"]
        else:
            swing_low.loc[ts] = row["Level"]

    addplots = [
        mpf.make_addplot(swing_high, type="scatter", marker="v", markersize=100, color="red"),
        mpf.make_addplot(swing_low, type="scatter", marker="^", markersize=100, color="green"),
    ]

    window_obs = obs.iloc[lo:hi].dropna(subset=["OB"])
    for ob_ts, ob_row in window_obs.iterrows():
        ob_pos = df.index.get_loc(ob_ts)
        end_pos = int(ob_row["MitigatedIndex"]) if ob_row["MitigatedIndex"] > 0 else hi - 1
        end_pos = min(end_pos, hi - 1)
        span_idx = df.index[ob_pos : end_pos + 1].intersection(window.index)
        top = pd.Series(index=window.index, dtype=float)
        bottom = pd.Series(index=window.index, dtype=float)
        top.loc[span_idx] = ob_row["Top"]
        bottom.loc[span_idx] = ob_row["Bottom"]
        color = "blue" if ob_row["OB"] == 1 else "darkorange"
        addplots.append(mpf.make_addplot(top, type="line", color=color, width=1.5))
        addplots.append(mpf.make_addplot(bottom, type="line", color=color, width=1.5))

    mpf.plot(
        window,
        type="candle",
        style="yahoo",
        addplot=addplots,
        title=title,
        warn_too_much_data=len(window) + 1,
        savefig=dict(fname=str(out_path), dpi=100),
    )

In [4]:
from datetime import datetime, timezone

summary = []

for timeframe in TIMEFRAMES:
    n_legs = LEGS_PER_TIMEFRAME[timeframe]
    n_points_needed = n_legs + 1  # N legs = N+1 swing points consecutivos

    df = store.get_ohlcv(SYMBOL, timeframe, datetime(2000, 1, 1, tzinfo=timezone.utc), datetime.now(timezone.utc))

    swings = detect_structural_swings(df)
    obs = smc.ob(df, swings, close_mitigation=False)
    obs.index = df.index
    swing_points = swings.dropna(subset=["HighLow"])
    swing_positions = [df.index.get_loc(ts) for ts in swing_points.index]

    if len(swing_positions) < n_points_needed:
        print(f"{timeframe}: solo {len(swing_positions)} swings confirmados, no alcanza para {n_legs} legs.")
        continue

    # Los últimos n_points_needed swings confirmados: desde hoy hacia atrás, no una fecha arbitraria.
    last_points = swing_positions[-n_points_needed:]
    # Solo estos puntos se marcan en la imagen -- el padding de la ventana no debe
    # traer de vuelta swings anteriores a los pedidos (eso pasó en el intento anterior).
    selected_swings = swing_points.loc[df.index[last_points]]

    first_pos, last_pos = last_points[0], last_points[-1]
    span = last_pos - first_pos
    pad = max(3, int(span * PAD_BARS_FRACTION))
    lo = max(0, first_pos - pad)
    hi = len(df)  # se extiende hasta la última vela disponible (hoy)

    out_path = OUTPUT_DIR / f"{SYMBOL}_{timeframe}.png"
    title = f"{SYMBOL} {timeframe} — últimos {n_legs} legs — {df.index[first_pos].date()} a {df.index[-1].date()}"
    render_structure_window(df, selected_swings, obs, lo, hi, title, out_path)
    summary.append({"timeframe": timeframe, "legs": n_legs, "bars": hi - lo, "path": str(out_path)})

    print(f"{timeframe}: {len(swing_positions)} swings totales, ventana de {hi - lo} velas, {len(selected_swings)} swings marcados.")

pd.DataFrame(summary)

1D: 276 swings totales, ventana de 11 velas, 4 swings marcados.


4H: 1603 swings totales, ventana de 29 velas, 7 swings marcados.


1H: 6580 swings totales, ventana de 26 velas, 7 swings marcados.


M30: 12828 swings totales, ventana de 26 velas, 7 swings marcados.


M15: 24779 swings totales, ventana de 29 velas, 7 swings marcados.


M5: 31879 swings totales, ventana de 30 velas, 7 swings marcados.


M1: 26085 swings totales, ventana de 39 velas, 7 swings marcados.


,timeframe,legs,bars,path
0,1D,3,11,C:\Users\isaac\Downloads\Projecto de trading I...
1,4H,6,29,C:\Users\isaac\Downloads\Projecto de trading I...
2,1H,6,26,C:\Users\isaac\Downloads\Projecto de trading I...
3,M30,6,26,C:\Users\isaac\Downloads\Projecto de trading I...
4,M15,6,29,C:\Users\isaac\Downloads\Projecto de trading I...
5,M5,6,30,C:\Users\isaac\Downloads\Projecto de trading I...
6,M1,6,39,C:\Users\isaac\Downloads\Projecto de trading I...
